# 🧠 MANATUABON PHASE 9: KAGGLE LORA FINE-TUNING (Unsloth Edition)

## NVIDIA Nemotron-3-Nano-30B Reasoning Optimization
Run this notebook on **Google Colab Pro** with an **A100 GPU (80GB)**.

Upload `synthetic_cot.jsonl` (generated by `kaggle_pipeline.py`) before running.

This notebook uses **Unsloth** which natively handles the hybrid Mamba-2 + MoE architecture.

In [ ]:
# 1. Install Unsloth + Dependencies
!pip install unsloth

In [ ]:
# 2. Load Model & Tokenizer via Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Nemotron-3-Nano-30B-A3B",
    max_seq_length=4096,
    dtype=None,         # Auto-detect (bf16 on A100)
    load_in_4bit=True,  # Unsloth handles 4-bit cleanly for hybrid models
)

print(f"\n✅ Model loaded successfully! Tokenizer vocab size: {len(tokenizer)}")

In [ ]:
# 3. Apply LoRA Adapter (Rank 32 max per Kaggle rules)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0,  # Unsloth optimized: dropout=0 is fastest
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # 60% less VRAM
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
# 4. Load the CoT Dataset
import os
from datasets import load_dataset

dataset_path = "synthetic_cot.jsonl"
if not os.path.exists(dataset_path):
    print("❌ ERROR: Please upload synthetic_cot.jsonl to the Colab workspace!")
else:
    dataset = load_dataset("json", data_files=dataset_path, split="train")

    def format_prompt(sample):
        convo = sample["conversations"]
        prompt = convo[0]["value"]
        response = convo[1]["value"]
        text = f"User: {prompt}\n\nAssistant: {response}{tokenizer.eos_token}"
        return {"text": text}

    dataset = dataset.map(format_prompt)

    # 90/10 Train/Eval split for Early Stopping
    dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
    train_data = dataset_split["train"]
    eval_data = dataset_split["test"]
    print(f"\n✅ Loaded {len(train_data)} train + {len(eval_data)} eval traces.")

In [ ]:
# 5. Configure Trainer with Early Stopping
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

output_dir = "./nemotron_reasoning_lora"

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=SFTConfig(
        output_dir=output_dir,
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=150,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=10,
        save_strategy="steps",
        save_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        optim="adamw_8bit",
        seed=42,
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# 6. Train!
print("\n🔥 Starting LoRA Fine-tuning with Early Stopping...")
trainer_stats = trainer.train()
print(f"\n✅ Training complete! Total steps: {trainer_stats.global_step}")

In [ ]:
# 7. Save the LoRA Adapter for Kaggle Submission
print(f"Saving LoRA adapter to {output_dir}...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("\n✅ LoRA adapter saved! Ready to zip and submit to Kaggle! 🏆")

In [ ]:
# 8. (Optional) Quick Inference Test
FastLanguageModel.for_inference(model)

test_prompt = "Find the next number in the sequence: 2, 6, 12, 20, 30, ?"
inputs = tokenizer(f"User: {test_prompt}\n\nAssistant:", return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.3)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))